# Load Cleaned E-Commerce Data into PostgreSQL

This notebook loads the cleaned Olist datasets generated by
03_data_cleaning.ipynb into PostgreSQL.

The cleaned datasets are stored in:

data/processed

The PostgreSQL database is:

ecommerce_ai_platform

In [ ]:
%pip install psycopg2-binary

In [1]:
import os
import io
import getpass
import warnings

import pandas as pd
import psycopg2
from psycopg2 import sql

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_ROOT = r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed data directory:")
print(PROCESSED_DATA_DIR)

Project root:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM

Processed data directory:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed


In [3]:
if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_ROOT}"
    )

if not os.path.exists(PROCESSED_DATA_DIR):
    raise FileNotFoundError(
        f"Processed data directory not found:\n{PROCESSED_DATA_DIR}"
    )

print("Project directory found successfully!")
print("Processed data directory found successfully!")

Project directory found successfully!
Processed data directory found successfully!


In [4]:
expected_files = [
    "customers_cleaned.csv",
    "geolocation_cleaned.csv",
    "order_items_cleaned.csv",
    "order_payments_cleaned.csv",
    "order_reviews_cleaned.csv",
    "orders_cleaned.csv",
    "products_cleaned.csv",
    "sellers_cleaned.csv",
    "category_translation_cleaned.csv"
]

missing_files = []

for file in expected_files:

    file_path = os.path.join(
        PROCESSED_DATA_DIR,
        file
    )

    if not os.path.exists(file_path):
        missing_files.append(file)

if missing_files:

    print("The following required files are missing:\n")

    for file in missing_files:
        print(file)

    raise FileNotFoundError(
        "\nRequired cleaned dataset files are missing."
    )

else:

    print("All required cleaned dataset files were found successfully!")

All required cleaned dataset files were found successfully!


In [5]:
print("Cleaned datasets available:\n")

for file in expected_files:

    file_path = os.path.join(
        PROCESSED_DATA_DIR,
        file
    )

    file_size_mb = (
        os.path.getsize(file_path)
        / (1024 * 1024)
    )

    print(
        f"{file:<40} "
        f"{file_size_mb:.2f} MB"
    )

Cleaned datasets available:

customers_cleaned.csv                    8.26 MB
geolocation_cleaned.csv                  42.03 MB
order_items_cleaned.csv                  14.33 MB
order_payments_cleaned.csv               5.47 MB
order_reviews_cleaned.csv                14.63 MB
orders_cleaned.csv                       19.19 MB
products_cleaned.csv                     3.14 MB
sellers_cleaned.csv                      0.16 MB
category_translation_cleaned.csv         0.00 MB


## PostgreSQL Connection Configuration

The password is entered securely using getpass and is never stored in the notebook code.

In [8]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "ecommerce_ai_db"
DB_USER = "postgres"

DB_PASSWORD = getpass.getpass(
    "Enter your PostgreSQL password: "
)

print("\nPostgreSQL connection details configured.")
print(f"Host: {DB_HOST}")
print(f"Port: {DB_PORT}")
print(f"Database: {DB_NAME}")
print(f"User: {DB_USER}")
print("Password: hidden")

Enter your PostgreSQL password:  ········



PostgreSQL connection details configured.
Host: localhost
Port: 5432
Database: ecommerce_ai_db
User: postgres
Password: hidden


In [9]:
try:

    connection = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    print("PostgreSQL connection successful!")

except Exception as error:

    print("PostgreSQL connection failed.")

    print("\nError details:")
    print(error)

    raise

PostgreSQL connection successful!


In [10]:
cursor = connection.cursor()

cursor.execute(
    "SELECT current_database(), current_user;"
)

database_name, username = cursor.fetchone()

print("Connected database:", database_name)
print("Connected user:", username)

cursor.close()

Connected database: ecommerce_ai_db
Connected user: postgres


In [11]:
datasets = {

    "customers": "customers_cleaned.csv",

    "geolocation": "geolocation_cleaned.csv",

    "order_items": "order_items_cleaned.csv",

    "order_payments": "order_payments_cleaned.csv",

    "order_reviews": "order_reviews_cleaned.csv",

    "orders": "orders_cleaned.csv",

    "products": "products_cleaned.csv",

    "sellers": "sellers_cleaned.csv",

    "category_translation": "category_translation_cleaned.csv"
}

print("Dataset-to-table mapping created successfully!")

for table_name, file_name in datasets.items():

    print(
        f"{table_name:<25} <- {file_name}"
    )

Dataset-to-table mapping created successfully!
customers                 <- customers_cleaned.csv
geolocation               <- geolocation_cleaned.csv
order_items               <- order_items_cleaned.csv
order_payments            <- order_payments_cleaned.csv
order_reviews             <- order_reviews_cleaned.csv
orders                    <- orders_cleaned.csv
products                  <- products_cleaned.csv
sellers                   <- sellers_cleaned.csv
category_translation      <- category_translation_cleaned.csv


In [12]:
dataframes = {}

for table_name, file_name in datasets.items():

    file_path = os.path.join(
        PROCESSED_DATA_DIR,
        file_name
    )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    dataframes[table_name] = df

    print(
        f"{table_name:<25} "
        f"{df.shape[0]:>8,} rows | "
        f"{df.shape[1]:>3} columns"
    )

customers                   99,441 rows |   5 columns
geolocation                738,332 rows |   5 columns
order_items                112,650 rows |   7 columns
order_payments             103,886 rows |   5 columns
order_reviews               99,224 rows |   7 columns
orders                      99,441 rows |  16 columns
products                    32,951 rows |  10 columns
sellers                      3,095 rows |   4 columns
category_translation            71 rows |   2 columns


In [13]:
for table_name, df in dataframes.items():

    print("\n" + "=" * 70)

    print(f"TABLE: {table_name}")

    print("=" * 70)

    print("Columns:")

    print(list(df.columns))

    print("\nData types:")

    print(df.dtypes)


TABLE: customers
Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

TABLE: geolocation
Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

TABLE: order_items
Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data types:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price

In [14]:
def convert_to_postgresql_type(dtype):

    dtype_string = str(dtype)

    if "int" in dtype_string:

        return "BIGINT"

    elif "float" in dtype_string:

        return "DOUBLE PRECISION"

    elif "bool" in dtype_string:

        return "BOOLEAN"

    elif "datetime" in dtype_string:

        return "TIMESTAMP"

    else:

        return "TEXT"

In [15]:
def create_table_sql(table_name, dataframe):

    column_definitions = []

    for column_name, dtype in dataframe.dtypes.items():

        postgres_type = convert_to_postgresql_type(dtype)

        column_definition = sql.SQL("{} {}").format(

            sql.Identifier(column_name),

            sql.SQL(postgres_type)

        )

        column_definitions.append(
            column_definition
        )

    query = sql.SQL(
        "CREATE TABLE IF NOT EXISTS {} ({})"
    ).format(

        sql.Identifier(table_name),

        sql.SQL(", ").join(column_definitions)

    )

    return query

In [16]:
cursor = connection.cursor()

for table_name, dataframe in dataframes.items():

    create_query = create_table_sql(
        table_name,
        dataframe
    )

    cursor.execute(create_query)

    print(
        f"Table ready: {table_name}"
    )

connection.commit()

cursor.close()

print("\nAll PostgreSQL tables created successfully!")

Table ready: customers
Table ready: geolocation
Table ready: order_items
Table ready: order_payments
Table ready: order_reviews
Table ready: orders
Table ready: products
Table ready: sellers
Table ready: category_translation

All PostgreSQL tables created successfully!


In [17]:
def clean_dataframe_for_postgresql(dataframe):

    df = dataframe.copy()

    df = df.where(
        pd.notnull(df),
        None
    )

    return df

In [18]:
def load_dataframe_to_postgresql(
    connection,
    dataframe,
    table_name
):

    dataframe = clean_dataframe_for_postgresql(
        dataframe
    )

    csv_buffer = io.StringIO()

    dataframe.to_csv(
        csv_buffer,
        index=False,
        header=False,
        na_rep="\\N"
    )

    csv_buffer.seek(0)

    cursor = connection.cursor()

    copy_query = sql.SQL(
        "COPY {} FROM STDIN WITH "
        "(FORMAT CSV, NULL '\\N')"
    ).format(

        sql.Identifier(table_name)

    )

    cursor.copy_expert(
        copy_query.as_string(connection),
        csv_buffer
    )

    connection.commit()

    cursor.close()

In [19]:
cursor = connection.cursor()

for table_name in datasets.keys():

    cursor.execute(
        sql.SQL(
            "TRUNCATE TABLE {}"
        ).format(
            sql.Identifier(table_name)
        )
    )

connection.commit()

cursor.close()

print("Existing table data cleared successfully!")

Existing table data cleared successfully!


In [20]:
for table_name, dataframe in dataframes.items():

    print(
        f"Loading {table_name}..."
    )

    load_dataframe_to_postgresql(
        connection=connection,
        dataframe=dataframe,
        table_name=table_name
    )

    print(
        f"{table_name} loaded successfully!"
    )

Loading customers...
customers loaded successfully!
Loading geolocation...
geolocation loaded successfully!
Loading order_items...
order_items loaded successfully!
Loading order_payments...
order_payments loaded successfully!
Loading order_reviews...
order_reviews loaded successfully!
Loading orders...
orders loaded successfully!
Loading products...
products loaded successfully!
Loading sellers...
sellers loaded successfully!
Loading category_translation...
category_translation loaded successfully!


In [21]:
cursor = connection.cursor()

row_counts = {}

for table_name in datasets.keys():

    cursor.execute(

        sql.SQL(
            "SELECT COUNT(*) FROM {}"
        ).format(

            sql.Identifier(table_name)

        )

    )

    row_count = cursor.fetchone()[0]

    row_counts[table_name] = row_count

cursor.close()

print("PostgreSQL row counts:\n")

for table_name, row_count in row_counts.items():

    print(
        f"{table_name:<25} "
        f"{row_count:>10,} rows"
    )

PostgreSQL row counts:

customers                     99,441 rows
geolocation                  738,332 rows
order_items                  112,650 rows
order_payments               103,886 rows
order_reviews                 99,224 rows
orders                        99,441 rows
products                      32,951 rows
sellers                        3,095 rows
category_translation              71 rows


In [22]:
comparison_results = []

for table_name, dataframe in dataframes.items():

    csv_row_count = len(dataframe)

    postgres_row_count = row_counts[table_name]

    status = (
        "MATCH"
        if csv_row_count == postgres_row_count
        else "MISMATCH"
    )

    comparison_results.append({

        "table_name": table_name,

        "csv_rows": csv_row_count,

        "postgres_rows": postgres_row_count,

        "status": status

    })

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,table_name,csv_rows,postgres_rows,status
0,customers,99441,99441,MATCH
1,geolocation,738332,738332,MATCH
2,order_items,112650,112650,MATCH
3,order_payments,103886,103886,MATCH
4,order_reviews,99224,99224,MATCH
5,orders,99441,99441,MATCH
6,products,32951,32951,MATCH
7,sellers,3095,3095,MATCH
8,category_translation,71,71,MATCH


In [23]:
if not all(
    comparison_df["status"] == "MATCH"
):

    raise ValueError(
        "Row count validation failed. "
        "At least one table has a mismatch."
    )

else:

    print(
        "SUCCESS: All CSV row counts match "
        "PostgreSQL row counts."
    )

SUCCESS: All CSV row counts match PostgreSQL row counts.


In [24]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        table_name
    FROM
        information_schema.tables
    WHERE
        table_schema = 'public'
    ORDER BY
        table_name;
    """
)

tables = cursor.fetchall()

cursor.close()

print("Tables currently available in PostgreSQL:\n")

for table in tables:

    print(table[0])

Tables currently available in PostgreSQL:

category_translation
customers
geolocation
order_items
order_payments
order_reviews
orders
products
sellers


In [25]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM
        information_schema.columns
    WHERE
        table_schema = 'public'
    ORDER BY
        table_name,
        ordinal_position;
    """
)

schema_information = cursor.fetchall()

cursor.close()

schema_df = pd.DataFrame(

    schema_information,

    columns=[
        "table_name",
        "column_name",
        "data_type"
    ]

)

schema_df

,table_name,column_name,data_type
0,category_translation,product_category_name,text
1,category_translation,product_category_name_english,text
2,customers,customer_id,text
3,customers,customer_unique_id,text
4,customers,customer_zip_code_prefix,bigint
...,...,...,...
56,products,product_category_name_english,text
57,sellers,seller_id,text
58,sellers,seller_zip_code_prefix,bigint
59,sellers,seller_city,text


In [26]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM
        information_schema.columns
    WHERE
        table_schema = 'public'
    ORDER BY
        table_name,
        ordinal_position;
    """
)

schema_information = cursor.fetchall()

cursor.close()

schema_df = pd.DataFrame(

    schema_information,

    columns=[
        "table_name",
        "column_name",
        "data_type"
    ]

)

print(
    "PostgreSQL schema verification completed successfully!"
)

schema_df.head(20)

PostgreSQL schema verification completed successfully!


,table_name,column_name,data_type
0,category_translation,product_category_name,text
1,category_translation,product_category_name_english,text
2,customers,customer_id,text
3,customers,customer_unique_id,text
4,customers,customer_zip_code_prefix,bigint
5,customers,customer_city,text
6,customers,customer_state,text
7,geolocation,geolocation_zip_code_prefix,bigint
8,geolocation,geolocation_lat,double precision
9,geolocation,geolocation_lng,double precision


In [27]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        COUNT(*)
    FROM
        public.orders;
    """
)

order_count = cursor.fetchone()[0]

cursor.close()

print(
    f"Orders available in PostgreSQL: "
    f"{order_count:,}"
)

Orders available in PostgreSQL: 99,441


In [28]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        COUNT(*)
    FROM
        public.order_items;
    """
)

order_item_count = cursor.fetchone()[0]

cursor.close()

print(
    f"Order items available in PostgreSQL: "
    f"{order_item_count:,}"
)

Order items available in PostgreSQL: 112,650


In [29]:
connection.close()

print(
    "PostgreSQL connection closed successfully."
)

PostgreSQL connection closed successfully.
